# 06 — Regression checks

| | |
|---|---|
| **Purpose** | Offline validators + curated MakerWorld URL catalog |
| **Website equivalent** | CI (`pytest`, `scripts/run_checks.sh`) |
| **Prev** | Run after `04_match_mcmaster.ipynb` or anytime |

Runs **no live network** by default. Set `RUN_LIVE_SCRAPE = True` to exercise MakerWorld URLs from `data/regression_urls.json` (slow; use `safe_scrape`).


In [ ]:
import subprocess
import sys

from backend.notebook_utils import REPO_ROOT, load_regression_catalog, print_pipeline_map

print_pipeline_map()
print()

result = subprocess.run(
    [sys.executable, str(REPO_ROOT / "scripts" / "run_checks.sh")],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
assert result.returncode == 0, f"run_checks.sh failed (exit {result.returncode})"


In [ ]:
from backend.notebook_utils import pick_sample_url, prepare_crawl_env, safe_scrape

RUN_LIVE_SCRAPE = False  # set True for manual MakerWorld regression

catalog = load_regression_catalog()
print(f"Curated URLs: {len(catalog.get('urls', []))}\n")
for entry in catalog.get("urls", []):
    flags = []
    if entry.get("expect_bom"):
        flags.append(f"bom={entry.get('expect_source', '?')}")
    else:
        flags.append("no bom expected")
    print(f"- {entry.get('label', entry['url'])}  [{', '.join(flags)}]")

if RUN_LIVE_SCRAPE:
    prepare_crawl_env(scraper="auto")
    for entry in catalog.get("urls", []):
        url = entry["url"]
        print(f"\n=== {entry.get('label', url)} ===")
        result = await safe_scrape(url, timeout_s=90)
        if result is None:
            print("  scrape failed")
            continue
        print(f"  title: {result.title}")
        print(f"  bom_source: {result.bom_source}")
        print(f"  embedded: {len(result.embedded_parts)}  file: {bool(result.bom_bytes)}")
else:
    print("\nLive scrape skipped — set RUN_LIVE_SCRAPE = True to crawl regression URLs.")
